# 🌳 Yggdrasil — Árvore de segmentação **unificada** (PD & LGD)

Tutorial do subpacote `yggdrasil.credit_risk.tree`. Uma **única classe**
(`TreeSegmenter`) e uma **única UI** (`TreeSegmenterUI`) atendem os dois problemas
de risco de crédito, escolhendo o comportamento por `task_type`:

| `task_type` | alvo | binning | IV | métricas | gráficos |
|---|---|---|---|---|---|
| `"classification"` (PD) | binário 0/1 | `OptimalBinning` | WoE (Siddiqi) | KS/AUC/Gini/Acurácia/F1 | ROC, KS, taxa-default, score |
| `"regression"` (LGD) | contínuo [0,1] | `ContinuousOptimalBinning` | IV contínuo | MAE/RMSE/R² | boxplot, histograma do alvo |

> Substitui as antigas classes separadas `SequentialPDSegmenter`/`SequentialLGDSegmenter`.
> A régua, a poda, o PSI/CSI, o backtest, a calibração, o `predict`/`to_pyspark`/
> `apply_spark`/`to_sql` e o `log_to_mlflow` são **os mesmos** nos dois modos.

**Instalação** (na raiz do repositório):
```bash
pip install -e ".[ui]"          # núcleo + interface interativa
pip install -e ".[ui,optuna]"   # + tuning bayesiano do ModelSegmenter
```
> No Databricks: `%pip install -e ".[ui]"` num cluster interativo DBR 13.0+.
> ⚠️ O `optbinning` 0.20 exige `scikit-learn<1.8` (já fixado no `pyproject`).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display
from yggdrasil.credit_risk.tree import TreeSegmenter

# --- base sintética de crédito: MESMAS features p/ os dois alvos -------------
# Features: comprometimento de renda, score de bureau (com faltantes), meses de
# relacionamento e tipo de garantia. dt_ref = safra (fora da modelagem).
# Amostras: DES (referência), OOT (out-of-time) e ESTABILIDADE (recente).
rng = np.random.default_rng(42)
GAR = {'alienacao': -0.55, 'aval': 0.15, 'fianca': 0.45, 'sem garantia': 1.05}

def gera(n, t0='2022-01-01'):
    dt_ref = pd.to_datetime(t0) + pd.to_timedelta(rng.integers(0, 365, n), unit='D')
    midx = (dt_ref.year.values - 2022) * 12 + (dt_ref.month.values - 1)
    comp = rng.beta(2.5, 3, n) * 0.8 + 0.1 + 0.004 * midx        # comprometimento migra no tempo
    score = np.clip(rng.normal(640, 95, n), 300, 900)
    score[rng.random(n) < 0.06] = np.nan                          # ~6% de score faltante
    relac = rng.integers(0, 120, n)
    gar = rng.choice(list(GAR), n, p=[0.50, 0.22, 0.18, 0.10]).astype(object)
    risco = (2.6 * (comp - 0.4) - 0.0045 * (np.nan_to_num(score, nan=640) - 640)
             - 0.004 * relac + np.array([GAR[g] for g in gar]))
    # alvo de CLASSIFICAÇÃO (default 0/1)
    p = 1 / (1 + np.exp(-(-2.15 + risco)))
    default = (rng.uniform(0, 1, n) < p).astype(float)
    # alvo de REGRESSÃO (LGD/severidade em [0,1])
    lgd = np.clip(0.45 + 0.18 * risco + rng.normal(0, 0.12, n), 0, 1)
    return pd.DataFrame({'comprometimento_renda': comp, 'score_bureau': score,
                         'meses_relacionamento': relac, 'tipo_garantia': gar,
                         'dt_ref': dt_ref, 'default': default, 'lgd': lgd})

des = gera(6000, '2022-01-01'); des['amostra'] = 'DES'
oot = gera(2500, '2023-07-01'); oot['amostra'] = 'OOT'
est = gera(1800, '2024-01-01'); est['amostra'] = 'ESTABILIDADE'
est['default'] = np.nan; est['lgd'] = np.nan          # público recente: ainda sem alvo
base = pd.concat([des, oot, est], ignore_index=True)

FEATS = ['comprometimento_renda', 'score_bureau', 'meses_relacionamento', 'tipo_garantia']
LABELS = {'comprometimento_renda': 'comprometimento de renda', 'score_bureau': 'score de bureau',
          'meses_relacionamento': 'meses de relacionamento', 'tipo_garantia': 'garantia'}
# views por tarefa (só o alvo relevante vira 'target'; o outro alvo fica de fora)
df_clf = base[FEATS + ['dt_ref', 'amostra', 'default']].rename(columns={'default': 'target'})
df_reg = base[FEATS + ['dt_ref', 'amostra', 'lgd']].rename(columns={'lgd': 'target'})
base.groupby('amostra', sort=False).agg(n=('default', 'size'),
                                        taxa_default=('default', 'mean'),
                                        lgd_medio=('lgd', 'mean')).round(4)

## 1) Classificação (PD) — `task_type="classification"`
A mesma API de sempre. O `fit_auto` cresce a árvore gulosa por **IV (WoE binário)**.

In [ ]:
seg_pd = TreeSegmenter(df_clf, target='target', task_type='classification',
                       sample_col='amostra', ref_sample='DES', date_col='dt_ref',
                       feature_labels=LABELS)
seg_pd.fit_auto(max_depth=3, min_iv=0.02)
display(seg_pd.leaves(with_psi=True).head(8))
display(seg_pd.metrics())                 # KS / AUC / Gini / Acurácia / F1 por amostra

In [ ]:
# discriminação visual + ranking de variáveis por IV
display(seg_pd.plot_roc())
display(seg_pd.plot_ks())
seg_pd.variable_iv('root')[['variavel', 'tipo', 'n_bins', 'iv', 'forca', 'pior_psi', 'psi_classificacao']]

## 2) Regressão (LGD) — `task_type="regression"`
**Mesma chamada, só muda o `task_type` e o alvo.** Agora o IV é contínuo e as
métricas são MAE/RMSE/R². Os gráficos específicos são boxplot e histograma do alvo.

In [ ]:
seg_lgd = TreeSegmenter(df_reg, target='target', task_type='regression',
                        sample_col='amostra', ref_sample='DES', date_col='dt_ref',
                        feature_labels=LABELS)
seg_lgd.fit_auto(max_depth=3, min_iv=0.01)
display(seg_lgd.leaves(with_psi=True).head(8))
display(seg_lgd.metrics())                # MAE / RMSE / R² por amostra
display(seg_lgd.plot_leaf_boxplots())     # dispersão do alvo por folha

## 3) Critério de split selecionável
Além do binning ótimo (`optbin`, multi-bin por IV), o `fit_auto`/`grow` aceitam um
**critério de split binário** (CART/CHAID) via `criterion=`:

- **classificação:** `gini`, `entropy`, `ks`, `iv`, `chi2`
- **regressão:** `variance`, `mae`, `ftest`

In [ ]:
for crit in ['optbin', 'ks', 'gini', 'chi2']:
    s = TreeSegmenter(df_clf, target='target', task_type='classification',
                      sample_col='amostra', ref_sample='DES', feature_labels=LABELS)
    s.fit_auto(max_depth=3, criterion=crit, verbose=False)
    nf = sum(v['is_leaf'] for v in s.segments.values())
    auc = s.metrics().query("amostra=='DES'")['AUC'].iloc[0]
    print(f"PD · criterion={crit:7} → {nf:2d} folhas · AUC(DES)={auc:.3f}")

for crit in ['optbin', 'variance', 'mae']:
    s = TreeSegmenter(df_reg, target='target', task_type='regression',
                      sample_col='amostra', ref_sample='DES', feature_labels=LABELS)
    s.fit_auto(max_depth=3, criterion=crit, verbose=False)
    nf = sum(v['is_leaf'] for v in s.segments.values())
    r2 = s.metrics().query("amostra=='DES'")['R2'].iloc[0]
    print(f"LGD · criterion={crit:8} → {nf:2d} folhas · R²(DES)={r2:.3f}")

## 4) Sugerir splits (TOP 3) e importância das variáveis
`suggest_splits(sid, top)` ranqueia as melhores variáveis para dividir uma folha,
com **nº de bins, IV, PSI por amostra (OOT/ESTABILIDADE), se passa no teste de
hipótese e o p-valor**. `feature_importance()` resume o ganho das variáveis que
**entraram** na árvore.

In [ ]:
print('PD — TOP 3 splits para a raiz:')
display(seg_pd.suggest_splits('root', top=3))
print('PD — importância das variáveis na árvore:')
display(seg_pd.feature_importance())
print('LGD — importância das variáveis na árvore:')
display(seg_lgd.feature_importance())

## 5) Auto-merge de folhas semelhantes
`auto_merge` funde folhas-irmãs com risco **estatisticamente indistinguível**
(p > α no teste entre adjacentes) — simplifica a régua sem perder separação.

In [ ]:
antes = sum(v['is_leaf'] for v in seg_pd.segments.values())
seg_pd.auto_merge(alpha=0.05)
depois = sum(v['is_leaf'] for v in seg_pd.segments.values())
print(f'PD: folhas {antes} → {depois} após auto-merge')

## 6) Exportar a régua: `predict`, **SQL (CASE WHEN)** e PySpark
A régua é aplicável em pandas (`predict`), gera **SQL copiável** (`to_sql`) e código
PySpark (`to_pyspark`). Faltantes são roteados para o bin `(faltante)` — nada é descartado.

In [ ]:
novos = gera(800).drop(columns=['default', 'lgd'])
display(seg_pd.predict(novos).head())             # segmento · nota · valor_regua
print(seg_pd.to_sql(table='carteira_credito'))    # cole no seu SQL

## 7) Comparar duas versões da árvore (`diff_trees`)
Compara a régua de duas árvores sobre a mesma base: **migração de notas**
(para onde as linhas vão), **concordância** e **métricas lado a lado**.

In [ ]:
seg_pd_v2 = TreeSegmenter(df_clf, target='target', task_type='classification',
                          sample_col='amostra', ref_sample='DES', feature_labels=LABELS)
seg_pd_v2.fit_auto(max_depth=2, verbose=False)     # versão mais rasa
d = seg_pd.diff_trees(seg_pd_v2)
print(f"concordância de notas: {d['concordancia']:.1%}")
display(d['resumo'])
display(d['migracao'])

## 8) Registrar no MLflow / Unity Catalog
`log_to_mlflow` registra a régua como modelo `pyfunc` e loga, **conforme o
`task_type`**:

- **params:** `variaveis` (que entraram na árvore), `profundidade`, `n_folhas`,
  `target`, `ref_sample`, ...
- **métricas:** `profundidade`, `n_variaveis`, **PSI por amostra** (`psi_OOT`,
  `psi_ESTABILIDADE`, ...) e as **métricas do modelo** — classificação:
  `ks_*`/`auc_*`/`gini_*`/`acuracia_*`/`f1_*`; regressão: `mae_*`/`rmse_*`/`r2_*`.
- **artefatos:** `folhas.csv`, `arvore.txt`, `regua.sql`, `regua_pyspark.py`,
  `regua.json`, `arvore.json` (para recarregar via `TreeSegmenter.load`).

```python
# PD
seg_pd.log_to_mlflow(experiment='/Users/voce/pd_seg',
                     registered_model_name='catalogo.schema.pd_segmentacao',
                     registry_uri='databricks-uc')
# LGD — mesma chamada
seg_lgd.log_to_mlflow(experiment='/Users/voce/lgd_seg',
                      registered_model_name='catalogo.schema.lgd_segmentacao',
                      registry_uri='databricks-uc')
```

## 9) Interface interativa unificada — `TreeSegmenterUI`
A mesma UI atende PD e LGD trocando só o `task_type`. 6 abas: **Construir**,
**Análise de variáveis**, **Diagnóstico**, **Validar & Exportar**, **Avançado**
(sugerir splits · auto-merge · importância · SQL · diff) e **Histórico**. No
Auto-fit há o seletor de **critério de split**; a discriminação ao vivo mostra
KS/AUC (clf) ou R² (reg).

```python
from yggdrasil.credit_risk.tree import TreeSegmenterUI

# PD (classificação)
ui = TreeSegmenterUI(df_clf, target='target', task_type='classification',
                     sample_col='amostra', ref_sample='DES', date_col='dt_ref',
                     feature_labels=LABELS, tree_samples=['DES', 'OOT'])
ui

# LGD (regressão) — só muda o task_type e o alvo
ui = TreeSegmenterUI(df_reg, target='target', task_type='regression',
                     sample_col='amostra', ref_sample='DES', date_col='dt_ref',
                     feature_labels=LABELS, tree_samples=['DES', 'OOT'])
ui
```